In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import basis, sigmam, sigmap, sigmaz, mesolve, expect

# 1. Define Ground |0> and Excited |1> states
ground_state = basis(2, 1)
excited_state = basis(2, 0)

# 2. Operators
sm = sigmam()  # Lowering operator (photon emission)
sp = sigmap()  # Raising operator (excitation)
sz = sigmaz()  # Phase monitoring

# 3. Define Noise Parameters
gamma_decay = 0.1   # Rate of spontaneous decay (loss)
gamma_phase = 0.05  # Rate of dephasing (background noise/decoherence)
n_thermal = 0.01    # Background thermal photon occupancy (Stray light)

# 4. Create Collapse Operators
c_ops = []
# Decay to environment
c_ops.append(np.sqrt(gamma_decay * (1 + n_thermal)) * sm)
# Excitation from background thermal light
c_ops.append(np.sqrt(gamma_decay * n_thermal) * sp)
# Dephasing noise (scattering/background interference)
c_ops.append(np.sqrt(gamma_phase) * sz)

# 5. Hamiltonian (emitter energy)
hbar = 1.0
omega = 1.0 * 2 * np.pi  # Frequency of the emitter (matching 1550nm)
H = 0.5 * hbar * omega * sz

# 6. Simulation Time (One pulse duration)
times = np.linspace(0, 100, 100)

# 7. Run the Simulation (Starting in the excited state to send a photon)
result = mesolve(H, excited_state, times, c_ops, [sm.dag() * sm])

# 8. Plot the probability of the system holding a photon over time
plt.figure(figsize=(8, 5))
plt.plot(times, result.expect[0], label='Probability of Photon Presence')
plt.title("Emitter Decay & Background Noise Simulation")
plt.xlabel("Time (ns)")
plt.ylabel("Photon Probability (Excited State)")
plt.grid(True)
plt.legend()
plt.show()

print(f"Final State Probability: {result.expect[0][-1]:.4f}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# SNSPD Parameters
efficiency = 0.8
dark_count_rate = 1e-7
dead_time = 20

# Input data
prob_array = result.expect[0] 
time_array = times
num_steps = len(time_array)

# Simulation
detector_output = []
last_click_at = -15

for i in range(num_steps):
    current_time = time_array[i]
    current_prob = prob_array[i]
    
    # If detector is in dead time, skip
    if current_time < (last_click_at + dead_time):
        click = 0
        
    else:
        # Signal Click probability
        p_signal = current_prob * efficiency
        
        # Dark Count probability
        p_dark = dark_count_rate
        
        # Combined probability of any click
        p_total = p_signal + p_dark
        
        # Roll the dice
        if np.random.random() < p_total:
            click = 1
            last_click_at = current_time
        else:
            click = 0
            
    detector_output.append(click)

# Results
plt.figure(figsize=(10, 4))
plt.step(time_array, detector_output, color='red', label='SNSPD Clicks (0 or 1)')
plt.fill_between(time_array, prob_array, alpha=0.3, label='Photon Signal Probability')
plt.title("SNSPD Detector Output")
plt.xlabel("Time (ns)")
plt.ylabel("Click Detected?")
plt.legend()
plt.show()

print(f"Total clicks detected: {sum(detector_output)}")